# Cell 1: Install Dependencies & Setup VITS Environment

In [1]:
# CELL 1
!git clone https://github.com/ylacombe/finetune-hf-vits.git

!pip install -r finetune-hf-vits/requirements.txt
!pip install datasets accelerate soundfile librosa Cython

# Use magic %cd commands to ensure setup.py runs in the exact correct context
%cd finetune-hf-vits/monotonic_align
!mkdir -p monotonic_align
!python setup.py build_ext --inplace
%cd ../..

Cloning into 'finetune-hf-vits'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 173 (delta 81), reused 74 (delta 74), pack-reused 78 (from 1)
Receiving objects: 100% (173/173), 1.24 MiB | 3.92 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/finetune-hf-vits/monotonic_align
Compiling core.pyx because it changed.
[1/1] Cythonizing core.pyx
/usr/local/lib/python3.12/dist-packages/Cython/Compiler/Main.py:381: FutureWarning: Cython directive 'language_level' not set, using '3str' for now (Py3). This has changed from earlier releases! File: /content/finetune-hf-vits/monotonic_align/core.pyx
  tree = Parsing.p_module(s, pxd, full_module_name)
performance hint: core.pyx:7:5: Exception check on 'maximum_path_each' will always require the GIL to be acquired.
Possible solutions:
	1. Declare 'maximum_path_each' as 'noexcept' if you control the definition and you're sure you don't wan

In [2]:
# !pip uninstall -y transformers
!pip install transformers==4.39.3 accelerate==0.27.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation: 

In [ ]:
# CELL 2
import os
import zipfile

# Download dataset
!gdown dvfgfdgdfgdfghdfgdfgdf

# Create target directory
os.makedirs('/content/dataset', exist_ok=True)

# Extract it
print("Extracting dataset...")
with zipfile.ZipFile('dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/dataset')

print("✅ Dataset successfully extracted to /content/dataset")

In [4]:
# CELL 3
import os
import json

root_dir = "/content/dataset"
metadata_lines =[]

print(f"🔍 Scanning {root_dir} for metadata.txt files...")

# Walk through dataset
for dirpath, _, filenames in os.walk(root_dir):
    for filename in filenames:
        # This will catch files like "e5c814f1-8ec1-fa67-8603-99120fe088ef-metadata.txt"
        if filename.endswith("metadata.txt"):
            txt_path = os.path.join(dirpath, filename)

            with open(txt_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line: continue
                    parts = line.split("|")

                    # Format: filename.wav | Text | Duration
                    if len(parts) >= 2:
                        wav_filename = parts[0]
                        text = parts[1]

                        # Ensure the .wav file exists right next to the txt file
                        full_wav_path = os.path.join(dirpath, wav_filename)
                        if os.path.exists(full_wav_path):
                            # HF needs the relative path from the dataset root
                            rel_wav_path = os.path.relpath(full_wav_path, root_dir)
                            metadata_lines.append({
                                "file_name": rel_wav_path,
                                "text": text
                            })

# Save metadata.jsonl explicitly into the dataset folder
jsonl_path = os.path.join(root_dir, "metadata.jsonl")
with open(jsonl_path, "w", encoding="utf-8") as f:
    for entry in metadata_lines:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"✅ Created {jsonl_path} with {len(metadata_lines)} mapped audio files.")

🔍 Scanning /content/dataset for metadata.txt files...
✅ Created /content/dataset/metadata.jsonl with 232 mapped audio files.


In [5]:
# CELL 4 (THE FINAL FIX)
import json
import os
from transformers import VitsModel, AutoTokenizer

print("📥 1. Downloading base model locally...")
model_id = "facebook/mms-tts-mon"
local_model_path = "./local_mms_model"
os.makedirs(local_model_path, exist_ok=True)

model = VitsModel.from_pretrained(model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model.save_pretrained(local_model_path)
tokenizer.save_pretrained(local_model_path)

print("⚙️ 2. Injecting missing feature extractor config...")
preprocessor_config = {
    "feature_extractor_type": "VitsFeatureExtractor",
    "feature_size": 80,
    "hop_length": 256,
    "max_wav_value": 32768.0,
    "n_fft": 1024,
    "padding_side": "right",
    "padding_value": 0.0,
    "return_attention_mask": False,
    "sampling_rate": 16000
}
with open(os.path.join(local_model_path, "preprocessor_config.json"), "w") as f:
    json.dump(preprocessor_config, f, indent=4)

print("📝 3. Generating Config...")
config = {
    "model_name_or_path": "./local_mms_model",
    "dataset_name": "/content/dataset",
    "audio_column_name": "audio",
    "text_column_name": "text",
    "train_split_name": "train",
    "output_dir": "./mms-mon-finetuned",
    "report_to": "none",                     # Disables W&B prompts safely
    "do_train": True,
    "do_eval": False,
    "learning_rate": 2e-5,
    "adam_beta1": 0.8,
    "adam_beta2": 0.99,
    "warmup_steps": 200,
    "max_grad_norm": 1.0,
    "fp16": True,
    "weight_duration": 1.0,
    "weight_kl": 1.5,
    "weight_mel": 35.0,
    "weight_disc": 3.0,
    "weight_gen": 1.0,
    "weight_fmaps": 1.0,
    "max_steps": 200000,
    "per_device_train_batch_size": 16,
    "gradient_accumulation_steps": 1,
    "dataloader_num_workers": 2,
    "logging_steps": 10,
    "save_steps": 200,
    "save_total_limit": 5,
    "push_to_hub": False
}
with open("finetune-hf-vits/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("🩹 4. Patching breaking changes from Transformers v5...")
script_path = "finetune-hf-vits/run_vits_finetuning.py"
with open(script_path, "r") as f:
    script_content = f.read()

# Patch A: Telemetry bug
script_content = script_content.replace("send_example_telemetry", "# send_example_telemetry")

# Patch B: Fix the overwrite_output_dir crash
script_content = script_content.replace(
    "training_args.overwrite_output_dir",
    "getattr(training_args, 'overwrite_output_dir', True)"
)

# Patch C: Fix the logging_dir crash
script_content = script_content.replace(
    "training_args.logging_dir",
    "(getattr(training_args, 'logging_dir', 'logs') or 'logs')"
)

with open(script_path, "w") as f:
    f.write(script_content)

print("✅ Setup complete! The trainer is now bulletproof against v5.0.0. Ready to train.")

📥 1. Downloading base model locally...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

Some weights of the model checkpoint at facebook/mms-tts-mon were not used when initializing VitsModel: ['flow.flows.0.wavenet.in_layers.0.weight_g', 'flow.flows.0.wavenet.in_layers.0.weight_v', 'flow.flows.0.wavenet.in_layers.1.weight_g', 'flow.flows.0.wavenet.in_layers.1.weight_v', 'flow.flows.0.wavenet.in_layers.2.weight_g', 'flow.flows.0.wavenet.in_layers.2.weight_v', 'flow.flows.0.wavenet.in_layers.3.weight_g', 'flow.flows.0.wavenet.in_layers.3.weight_v', 'flow.flows.0.wavenet.res_skip_layers.0.weight_g', 'flow.flows.0.wavenet.res_skip_layers.0.weight_v', 'flow.flows.0.wavenet.res_skip_layers.1.weight_g', 'flow.flows.0.wavenet.res_skip_layers.1.weight_v', 'flow.flows.0.wavenet.res_skip_layers.2.weight_g', 'flow.flows.0.wavenet.res_skip_layers.2.weight_v', 'flow.flows.0.wavenet.res_skip_layers.3.weight_g', 'flow.flows.0.wavenet.res_skip_layers.3.weight_v', 'flow.flows.1.wavenet.in_layers.0.weight_g', 'flow.flows.1.wavenet.in_layers.0.weight_v', 'flow.flows.1.wavenet.in_layers.1.wei

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

⚙️ 2. Injecting missing feature extractor config...
📝 3. Generating Config...
🩹 4. Patching breaking changes from Transformers v5...
✅ Setup complete! The trainer is now bulletproof against v5.0.0. Ready to train.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
# # CELL 5 (FINAL FIXES)
# import os

# # 1. Disable W&B so it stops pausing and asking for an account
# os.environ["WANDB_DISABLED"] = "true"

# # 2. Patch the DataCollator in the training script to fix the "NoneType" crash
# script_path = "finetune-hf-vits/run_vits_finetuning.py"
# with open(script_path, "r") as f:
#     script = f.read()

# # Replace the bugged BatchEncoding line with a safe dictionary filter
# script = script.replace(
#     "batch = BatchEncoding(batch)",
#     "batch = {k: v for k, v in batch.items() if v is not None}"
# )

# with open(script_path, "w") as f:
#     f.write(script)

# print("✅ Disabled W&B and patched the Batch GPU bug. Ready to launch!")

# Resampling Audio

In [7]:
import os
import librosa
import soundfile as sf
import numpy as np
from tqdm.notebook import tqdm

raw_dir = "/content/dataset/dataset-raw"

print(f"🎧 Safely downsampling and normalizing audio in {raw_dir} to 16,000 Hz Mono...")

wav_files = [f for f in os.listdir(raw_dir) if f.endswith(".wav")]

for wav_file in tqdm(wav_files):
    file_path = os.path.join(raw_dir, wav_file)

    # 1. Load the audio (downsamples to 16kHz Mono using high-quality filter)
    audio, sr = librosa.load(file_path, sr=16000, mono=True)

    # 2. Peak Normalization: Boost the volume to the maximum safe level
    # This prevents the 16kHz audio from sounding weak, dull, or quiet
    audio = librosa.util.normalize(audio)

    # 3. Overwrite the original file with the clean, loud 16kHz version
    sf.write(file_path, audio, 16000, subtype='PCM_16')

print("✅ All audio safely converted, normalized, and ready for training!")

🎧 Safely downsampling and normalizing audio in /content/dataset/dataset-raw to 16,000 Hz Mono...


  0%|          | 0/232 [00:00<?, ?it/s]

✅ All audio safely converted, normalized, and ready for training!


In [8]:
# CELL 5 (CLEAN RESET)
import urllib.request
import json

# 1. Download a fresh, uncorrupted copy of the training script
url = "https://raw.githubusercontent.com/ylacombe/finetune-hf-vits/main/run_vits_finetuning.py"
urllib.request.urlretrieve(url, "finetune-hf-vits/run_vits_finetuning.py")

# 2. Safely patch ONLY the telemetry bug (no hacky string replacements this time)
with open("finetune-hf-vits/run_vits_finetuning.py", "r") as f:
    script = f.read()

script = script.replace(
    "from transformers.utils import send_example_telemetry",
    "# from transformers.utils import send_example_telemetry"
)
script = script.replace("send_example_telemetry(", "# send_example_telemetry(")

with open("finetune-hf-vits/run_vits_finetuning.py", "w") as f:
    f.write(script)

# 3. Officially disable W&B in the config to stop the prompts
config_path = "finetune-hf-vits/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

config["report_to"] = "none"

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print("✅ Clean reset complete. Accelerate downgraded and script safely patched.")

✅ Clean reset complete. Accelerate downgraded and script safely patched.


In [ ]:
# CELL 6
!rm -rf ./mms-mon-finetuned
!WANDB_DISABLED=true accelerate launch --num_processes=1 --num_machines=1 --mixed_precision="no" --dynamo_backend="no" finetune-hf-vits/run_vits_finetuning.py finetune-hf-vits/config.json

# Inference

In [10]:
from transformers import VitsModel, AutoTokenizer, VitsConfig
from safetensors.torch import load_file
import torch
import torch.nn.functional as F
import numpy as np
from IPython.display import Audio, display
import os

# Auto-find latest checkpoint
output_dir = "./mms-mon-finetuned"
checkpoints = sorted(
    [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
checkpoint_path = os.path.join(output_dir, checkpoints[-1])
config_path = output_dir
print(f"🔊 Running inference on: {checkpoint_path}")

# Load raw weights
raw = load_file(f"{checkpoint_path}/model.safetensors")
if os.path.exists(f"{checkpoint_path}/model_1.safetensors"):
    raw.update(load_file(f"{checkpoint_path}/model_1.safetensors"))

# Convert weight_g + weight_v → weight
converted = {}
for key in raw:
    if key.endswith(".weight_g"):
        base = key[:-len(".weight_g")]
        key_v = base + ".weight_v"
        if key_v in raw:
            g = raw[key]
            v = raw[key_v]
            v_norm = F.normalize(v.view(v.shape[0], -1), dim=1).view_as(v)
            converted[base + ".weight"] = g * v_norm
    elif key.endswith(".weight_v"):
        continue
    else:
        converted[key] = raw[key]

print(f"✅ Converted {len([k for k in raw if k.endswith('.weight_g')])} weight_norm pairs")
print(f"✅ Final state dict has {len(converted)} keys")

config = VitsConfig.from_pretrained(config_path)
model = VitsModel(config)
missing, unexpected = model.load_state_dict(converted, strict=False)
print(f"Missing keys: {len(missing)}")
print(f"Unexpected keys: {len(unexpected)}")
model.eval()

tokenizer = AutoTokenizer.from_pretrained(config_path)
text = "Өнөөдөр гадаа хасах хорин хэм хүйтэн байна."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform

waveform_np = output.squeeze().cpu().numpy()
print("Max amplitude:", np.abs(waveform_np).max())
print("NaN present:", np.isnan(waveform_np).any())

display(Audio(waveform_np, rate=model.config.sampling_rate))

🔊 Running inference on: ./mms-mon-finetuned/checkpoint-200
✅ Converted 121 weight_norm pairs
✅ Final state dict has 836 keys
Missing keys: 0
Unexpected keys: 74
Max amplitude: 0.65125585
NaN present: False


# Resume Training

In [ ]:
import os
import json

output_dir = "./mms-mon-finetuned"
checkpoints = sorted(
    [d for d in os.listdir(output_dir) if d.startswith("checkpoint-")],
    key=lambda x: int(x.split("-")[1])
)
latest = os.path.join(output_dir, checkpoints[-1])
print(f"▶️ Resuming from: {latest}")

# Inject resume path directly into config.json
config_path = "finetune-hf-vits/config.json"
with open(config_path, "r") as f:
    config = json.load(f)

config["resume_from_checkpoint"] = latest

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

# Launch normally — no extra CLI args needed
!WANDB_DISABLED=true accelerate launch \
    --num_processes=1 \
    --num_machines=1 \
    --mixed_precision=fp16 \
    --dynamo_backend=no \
    ./finetune-hf-vits/run_vits_finetuning.py \
    ./finetune-hf-vits/config.json